# 02 — Split-half reliability + intergroup bootstrap

Generates synthetic two-group recognition memory data with a **known true intergroup correlation** $\rho$. Verifies:

1. `estimate_split_half_flexible` returns a sensible within-group reliability.
2. `bootstrap_intergroup_correlation_sem` returns a CI that covers the true $\rho$.
3. Attenuation correction inflates raw r toward $\rho$.

Vary `rho_true`, `n_a`, `n_b`, `n_items`, and `target_rel` to see how each affects the result.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd(); ROOT = HERE.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from python import (
    estimate_split_half_flexible,
    split_half_sb,
    calculate_split_half_reliability,
    bootstrap_intergroup_correlation_sem,
)

## Synthetic data generator

Each item has a latent rate in each group; the two groups' rates are drawn jointly from a bivariate normal with correlation `rho_true`. Each participant's per-item response is a Bernoulli draw around a noisy version of that latent rate (the noise std controls within-group reliability).

In [ ]:
def synth_two_groups(rho_true=0.6, n_a=25, n_b=25, n_items=60,
                     base_rate=0.7, item_spread=0.15, target_rel=0.7,
                     seed=0):
    rng = np.random.default_rng(seed)
    R = np.array([[1.0, rho_true], [rho_true, 1.0]])
    Z = rng.multivariate_normal([0, 0], R, size=n_items)
    item_rates = np.clip(base_rate + item_spread * Z, 0.02, 0.98)

    def make_group(n, idx):
        if target_rel >= 1 or target_rel <= 0:
            sig_e = 0.0
        else:
            sig_e = item_spread * np.sqrt(n * (1 - target_rel) / target_rel)
        E = sig_e * rng.standard_normal((n, n_items))
        P = np.clip(item_rates[:, idx][None, :] + E, 0.01, 0.99)
        return (rng.random((n, n_items)) < P).astype(float)

    return make_group(n_a, 0), make_group(n_b, 1), item_rates

XA, XB, item_rates = synth_two_groups(rho_true=0.6, seed=1)
print('XA:', XA.shape, ' XB:', XB.shape, ' XA mean:', XA.mean().round(3))

## Split-half reliability

`point_r` is the **median** of per-split correlations (see STATS.md §2). Spearman-Brown corrects the half-set reliability to the full-set reliability.

In [ ]:
point_r_a, std_r_a, rs_a = estimate_split_half_flexible(
    XA, n_splits=2000, split_dim=1, corr_type='Spearman',
    rng=np.random.default_rng(2),
)
sb_a = (2 * point_r_a) / (1 + point_r_a)
print(f'Group A: half-set r (median) = {point_r_a:.3f} +/- {std_r_a:.3f} | SB = {sb_a:.3f}')

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(rs_a[np.isfinite(rs_a)], bins=30, color='#4060b0', alpha=0.75)
ax.axvline(point_r_a, color='k', lw=1.5, label=f'median = {point_r_a:.3f}')
ax.set_xlabel('Per-split r'); ax.set_ylabel('count'); ax.legend()
ax.set_title('Group A split-half distribution')
plt.tight_layout(); plt.show()

## Intergroup bootstrap

We treat `calculate_split_half_reliability` as the canonical way to package matrices + items + SB into the struct expected by the bootstrap. The 'fas' matrix is unused here so we pass NaNs.

In [ ]:
items = np.array([f'item{i:03d}' for i in range(XA.shape[1])], dtype=object)

outs_a = calculate_split_half_reliability(
    XA, np.full_like(XA, np.nan), items, n_splits=500, split_dim=1,
    rng=np.random.default_rng(3),
)
outs_b = calculate_split_half_reliability(
    XB, np.full_like(XB, np.nan), items, n_splits=500, split_dim=1,
    rng=np.random.default_rng(4),
)
print(f'SB_A = {outs_a.sb_hit:.3f}   SB_B = {outs_b.sb_hit:.3f}')

res = bootstrap_intergroup_correlation_sem(
    outs_a, outs_b, trial_type='hit',
    n_boot=2000, min_resp=2, use_spearman=True,
    reliability_mode='fixed', correct_atten=True,
    rng=np.random.default_rng(5), verbose=True,
)

In [ ]:
print(f'\nRESULTS')
print(f'  point r (raw)        = {res.point_raw:.3f}')
print(f'  95% CI (raw)         = [{res.ci_raw[0]:.3f}, {res.ci_raw[1]:.3f}]')
print(f'  median boot (raw)    = {res.median_boot_raw:.3f}')
print(f'  point r (corrected)  = {res.point_corr:.3f}')
print(f'  95% CI (corrected)   = [{res.ci_corr[0]:.3f}, {res.ci_corr[1]:.3f}]')
print(f'\nTRUE rho = 0.6   <- corrected point should sit near this')

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(res.r_boot_raw,  bins=30, color='#4060b0', alpha=0.5, label='raw')
ax.hist(res.r_boot_corr, bins=30, color='#b04040', alpha=0.5, label='atten-corrected')
ax.axvline(0.6, color='k', lw=1.5, ls='--', label='true rho = 0.6')
ax.set_xlabel('bootstrap r'); ax.set_ylabel('count'); ax.legend()
plt.tight_layout(); plt.show()

**What to look for.**

- Raw bootstrap distribution is biased toward 0 by measurement noise (true $\rho=0.6$, observed raw $r$ is lower).
- Attenuation-corrected distribution should be roughly centered on the true $\rho$.
- CI for the corrected $r$ should cover 0.6.

Try setting `target_rel=0.4` in the generator and rerunning — the corrected distribution gets wider (low reliability means a large divisor in $r/\sqrt{\rho_A\rho_B}$), and you'll see the $\pm 1$ clamp bite occasionally.